**SAVED_SECTION:10**

---

## Notebook Complete! 🎉

This notebook covered the complete RAGAS evaluation framework implementation:
- Theory foundation and metric understanding
- Golden test set creation and management
- RAGAS evaluation with LLM-as-judge
- Domain-aware thresholds
- Cost analysis and when NOT to use
- Common failures and fixes
- Production deployment considerations
- Decision framework

**Next steps:**
1. Set up your OpenAI API key in `.env`
2. Create your domain-specific golden test set (100+ questions)
3. Integrate with your RAG system
4. Run nightly evaluations
5. Monitor for regressions

**Further reading:**
- Module 8.2: A/B Testing for RAG Systems
- Module 8.3: Regression Testing in CI/CD
- Module 8.4: Human-in-the-Loop Evaluation

Good luck with your RAG evaluation! 🚀

In [ ]:
# Summary of key takeaways
print("="*60)
print("MODULE 8.1: RAGAS EVALUATION FRAMEWORK - KEY TAKEAWAYS")
print("="*60)

takeaways = [
    ("Four Metrics", "Faithfulness, Relevancy, Precision, Recall - each catches different failures"),
    ("Golden Set", "100-300 questions, 40-80 hours to create, foundation of evaluation"),
    ("Cost Reality", "$80-200/month for daily runs (100 questions with GPT-3.5)"),
    ("When NOT to use", "<100 queries/month, pre-PMF, creative answers"),
    ("Alternatives", "Manual review, ground truth comparison, user feedback"),
    ("Domain Thresholds", "Compliance needs ≥0.90 faithfulness, adjust by criticality"),
    ("Resilient Design", "Batching, checkpointing, rate limiting for production"),
    ("Regression Detection", ">5% drop = alert, track baselines, avoid false positives")
]

for i, (key, detail) in enumerate(takeaways, 1):
    print(f"\\n{i}. {key}")
    print(f"   → {detail}")

print("\\n" + "="*60)
print("Next: Module 8.2 - A/B Testing for RAG Systems")
print("="*60)

# Expected: Prints summary of 8 key takeaways

**SAVED_SECTION:9**

---

## Section 10: Decision Card - Quick Reference Guide

### ✅ BENEFIT
Systematic regression detection with four metrics (faithfulness, relevance, precision, recall) that catch issues before users complain. Provides statistical confidence via automated nightly evaluation of 100+ test cases. Industry-standard approach that scales evaluation from 2 hours manual review to 5 minutes automated.

### ❌ LIMITATION
Requires 40-80 hours to create quality golden test set of 100+ questions. LLM-as-judge has 10-15% disagreement with human reviewers on edge cases. Doesn't catch domain-specific correctness (e.g., 'Is this the right regulation?'). High ongoing cost: $2-5 per evaluation means $60-150/month for daily runs.

### 💰 COST
**Time:** 40-80 hours golden set creation, 12 hours setup, 2-4 hours/month maintenance  
**Money:** $80-200/month operational cost (API: $60-150, MLflow: $10-30, storage: $10-20)  
**Complexity:** 4 new components (RAGAS, golden set manager, pipeline, MLflow), 800+ lines of code

### 🤔 USE WHEN
- You have **1000+ production queries/month** with stable query patterns
- Budget allows **$80-200/month** evaluation cost
- Query types are **factual/compliance-focused** (not creative)
- You need to **detect regressions systematically**
- Team has **40-80 hours** for initial golden set creation
- Alternative approaches (manual review, ground truth comparison) don't scale

### 🚫 AVOID WHEN
- **<100 queries/month** → Use manual review
- **Pre-product/market fit** → Query patterns unstable
- **Creative/subjective answers** → Use user feedback
- **Budget <$1000/month total** → Evaluation overhead too high (>20%)
- **Answers require domain-specific correctness verification** (medical, legal) that RAGAS can't validate

---

**Save this card—you'll reference it when your team debates whether to implement automated evaluation.**

**SAVED_SECTION:8**

---

## Section 9: Production Considerations

### Scaling Concerns

**Evaluation frequency trade-offs:**
- **Nightly (recommended):** Catches regressions within 24 hours, $60-150/month
- **Per-deployment:** Only on changes, more cost-effective but misses gradual drift
- **Weekly:** Budget-friendly ($15-40/month) but slower detection

**Large test set handling:**
- 100 questions: 5-8 minutes, standard evaluation
- 300 questions: 15-20 minutes, use batching
- 1000+ questions: 1+ hour, use distributed evaluation (not covered here)

### Cost Breakdown (Monthly for 100-question golden set)

| Component | Daily Runs | Weekly Runs | Notes |
|-----------|------------|-------------|-------|
| OpenAI API | $60-150 | $15-40 | GPT-3.5-Turbo |
| MLflow | $10-30 | $10-30 | Optional |
| Storage | $10-20 | $5-10 | Results history |
| **Total** | **$80-200** | **$30-80** | |

### Monitoring Requirements

**Track these metrics:**
1. **Evaluation success rate:** Should be >95%
2. **Evaluation duration:** Should be consistent (±20%)
3. **Cost per evaluation:** Monitor for API price changes
4. **False positive rate:** <10% (unnecessary alerts)

### Production Deployment Checklist

- [ ] Golden set created with 100+ questions
- [ ] Representative of production query distribution (verified)
- [ ] Domain-specific thresholds configured
- [ ] Baseline established from stable version
- [ ] Batched evaluation with checkpointing enabled
- [ ] Regression alerts configured (>5% drop)
- [ ] Cost monitoring and budget alerts set
- [ ] Results history retention policy defined (30-90 days)
- [ ] Runbook for common failures documented

### Integration with Level 1 M2.3 Monitoring

RAGAS evaluation should integrate with your existing monitoring:

```python
# Prometheus metrics export
from prometheus_client import Gauge

ragas_faithfulness = Gauge('ragas_faithfulness', 'RAGAS faithfulness score')
ragas_relevancy = Gauge('ragas_answer_relevancy', 'RAGAS answer relevancy score')
ragas_precision = Gauge('ragas_context_precision', 'RAGAS context precision score')
ragas_recall = Gauge('ragas_context_recall', 'RAGAS context recall score')

# Update after each evaluation
def export_metrics(scores):
    ragas_faithfulness.set(scores['faithfulness'])
    ragas_relevancy.set(scores['answer_relevancy'])
    ragas_precision.set(scores['context_precision'])
    ragas_recall.set(scores['context_recall'])
```

This allows Grafana dashboards to track evaluation metrics alongside production metrics.

In [ ]:
# Demonstrate resilient evaluation with batching
# In production, you would use this for large golden sets (100+ questions)

print("💪 Resilient Evaluation Features:\\n")
print("✓ Batched processing (20 questions/batch)")
print("✓ Checkpoint recovery (resume from failures)")
print("✓ Rate limiting (2s delay between batches)")
print("✓ Progress tracking\\n")

print("Example usage:")
print("""
resilient = ResilientEvaluator(batch_size=20)
results = resilient.evaluate_with_batching(
    questions=questions,
    generated_answers=answers,
    retrieved_contexts=contexts,
    ground_truths=ground_truths,
    checkpoint_name="production_eval"
)
""")

print("\\nIf evaluation fails mid-way, it will resume from last checkpoint automatically.")

# Expected: Shows features of resilient evaluator

---

## Section 8: Common Failures (Five Ways RAGAS Breaks)

Understanding failure modes is critical for debugging production issues. Here are the five most common failures from the source material:

### Failure #1: Golden Set Quality Issues (Biased Test Set)

**The problem:**
Your golden set only tests one query pattern. Production users ask different types of questions. RAGAS scores look great, but users complain.

**Example:**
- Golden set: 100 questions like "What is GDPR Article X?"
- Production queries: "How does GDPR Article 5 apply to healthcare employee records when consent wasn't obtained?"
- Your test scores: 0.89 faithfulness - looks great!
- Production reality: 43% of queries failing

**The fix:**
- Analyze production query distribution
- Create representative golden set matching production patterns (±10%)
- Include all query types: definitions, explanations, procedural, comparisons, recommendations
- Refresh quarterly from production logs

### Failure #2: RAGAS Metric Interpretation Errors

**The problem:**
You're averaging metrics when faithfulness is binary—either you hallucinate or you don't. A 0.45 faithfulness means 55% of statements are not grounded. That's catastrophic for compliance.

**Example:**
- Scores: faithfulness=0.45, relevancy=0.82, precision=0.76, recall=0.71
- You think: "Average = 0.685, that's passing!"
- Reality: You're hallucinating on 55% of statements

**The fix:**
- Use domain-specific thresholds (not averages)
- Compliance domains: faithfulness ≥ 0.90 (hard requirement)
- Treat faithfulness as binary: pass/fail, not gradual
- Weight metrics by criticality for your domain

### Failure #3: Evaluation Pipeline Timeouts (Large Test Sets)

**The problem:**
Evaluating 300 questions × 4 metrics = 1200 API calls. If any times out, entire evaluation fails and you lose progress.

**The fix:**
- Use batched evaluation (20 questions per batch)
- Implement checkpointing (resume from failures)
- Add rate limiting (2 second delay between batches)
- Our `ResilientEvaluator` handles this automatically

### Failure #4: Performance Tracking Gaps (Missing Baselines)

**The problem:**
You run evaluation, get scores, but have no context. Is 0.75 good? Better than yesterday? Without baselines, scores are meaningless.

**The fix:**
- Establish baseline on first run
- Track deltas vs. baseline
- Flag regressions >5% drop
- Use `EvaluationPipeline` for automatic baseline tracking

### Failure #5: False Regression Detection (Normal Variance)

**The problem:**
LLM-as-judge has natural variance (±3-5%). You flag every small drop as regression, creating alert fatigue.

**The fix:**
- Only alert on drops >5% (above natural variance)
- Require 2-3 consecutive drops before alerting
- Track trends, not individual runs
- Use statistical significance testing for large sets

**SAVED_SECTION:7**

---

## Section 7: When NOT to Use RAGAS (Three Critical Scenarios)

### Scenario 1: Pre-Product/Market Fit (Query Patterns Unstable)

**The problem:**
You're still figuring out what users actually want. Query patterns change weekly. Your carefully crafted 100-question golden set becomes obsolete in 2 weeks.

**Signs this is you:**
- Launched <3 months ago
- User query distribution shifts >20% week-over-week
- Still iterating on core use cases
- <50 daily active users

**What to do instead:**
- Manual review of ALL queries (you don't have many yet)
- Track query patterns, wait for stability
- Build golden set when query distribution stabilizes
- Estimated time to stability: 3-6 months post-launch

### Scenario 2: Low Query Volume (<100/month)

**The problem:**
You're spending $80-200/month to evaluate a system that handles 50 queries/month. Evaluation costs exceed production costs. ROI doesn't make sense.

**The math:**
- 50 queries/month production: ~$5-15/month in LLM costs
- RAGAS evaluation: $80-200/month
- You're spending 10x on evaluation vs. production

**What to do instead:**
- Manual review: 1-2 hours/week
- Ground truth comparison for deterministic questions
- User feedback collection
- Revisit RAGAS when you hit 500+ queries/month

### Scenario 3: Answers Without Verifiable Ground Truth

**The problem:**
Your domain requires creativity, opinion, or subjective judgment. RAGAS compares to ground truth, but there's no single correct answer.

**Examples:**
- Creative writing assistants
- Brainstorming tools
- Opinion/recommendation systems
- Open-ended customer support

**What RAGAS will do:**
- Penalize creative but valid answers
- Reward answers that match ground truth verbatim
- Give misleading scores (low scores on good answers)

**What to do instead:**
- A/B testing with user engagement metrics
- User satisfaction surveys
- Task completion rates
- Qualitative feedback collection

**SAVED_SECTION:6**

---

## Section 6: Alternative Solutions

When RAGAS is too heavy or expensive, consider these alternatives:

### Alternative 1: Manual Review with Sampling (Small Scale)
**Best for:** <100 queries/month

**Approach:**
- Review 10-20 randomly sampled responses weekly
- Use rubric: factual accuracy, relevance, completeness
- Track issues in spreadsheet
- Cost: 1-2 hours/week of expert time

**Trade-offs:**
- ✅ Zero API costs
- ✅ Catches domain-specific issues RAGAS misses
- ❌ Not systematic, prone to bias
- ❌ Doesn't scale beyond 100 queries/month

### Alternative 2: Ground Truth Comparison (Simple Automated)
**Best for:** Questions with deterministic answers

**Approach:**
```python
def simple_evaluation(answer: str, ground_truth: str) -> float:
    # Exact match or embedding similarity
    answer_emb = get_embedding(answer)
    truth_emb = get_embedding(ground_truth)
    return cosine_similarity(answer_emb, truth_emb)
```

**Trade-offs:**
- ✅ Fast, cheap (~$0.10 per 100 questions)
- ✅ Works for factual lookups
- ❌ Misses hallucinations
- ❌ Poor for multi-part answers

### Alternative 3: Production Feedback (User-Driven)
**Best for:** Customer-facing systems with clear user actions

**Approach:**
- Track thumbs up/down on answers
- Monitor query refinement rate (user asks follow-up)
- Track click-through on retrieved documents

**Trade-offs:**
- ✅ Real user signal, not synthetic
- ✅ Zero evaluation cost
- ❌ Lagging indicator (users already frustrated)
- ❌ Biased (happy users don't give feedback)

### Decision Framework: Which Evaluation Approach?

| Scale | Budget | Domain | Recommended Approach |
|-------|--------|--------|---------------------|
| <100 queries/month | Any | Any | Manual review |
| 100-1000/month | <$1000/month | Factual | Ground truth comparison |
| 100-1000/month | >$1000/month | Factual | RAGAS (weekly runs) |
| 1000+/month | >$5000/month | Any | RAGAS (daily runs) + User feedback |

**SAVED_SECTION:5**

---

## Section 5: Reality Check - What RAGAS Actually Costs You

### Time Investment
- **Golden set creation:** 40-80 hours for 100-300 questions
- **Setup and integration:** 8-12 hours
- **Maintenance:** 2-4 hours/month (updating golden set, analyzing results)

### Money Cost
- **API costs:** $2-5 per 100 questions with GPT-3.5-Turbo
  - Daily runs (100 questions): ~$60-150/month
  - Weekly runs: ~$15-40/month
- **GPT-4 is 15x more expensive:** ~$30 per 100 questions
- **MLflow hosting (optional):** $10-30/month

### Monthly Operational Cost
For a production system running daily evaluation on 100 questions:
- API: $60-150
- MLflow: $10-30 (optional)
- Storage: $10-20
- **Total: $80-200/month**

### When NOT to use RAGAS (critical decision points)
1. **<100 queries/month** → Use manual review instead
2. **Pre-product/market fit** → Query patterns unstable, golden set obsolete quickly
3. **Budget <$1000/month total** → Evaluation overhead too high (>20% of budget)
4. **Creative/subjective answers** → Use user feedback metrics instead

In [ ]:
# Step 3: Domain-aware evaluation
domain_evaluator = DomainAwareEvaluator(domain="compliance")

# Simulate scores (in production, these come from RAGAS evaluation)
mock_scores = {
    "faithfulness": 0.85,        # Below compliance threshold (0.90)
    "answer_relevancy": 0.78,
    "context_precision": 0.72,
    "context_recall": 0.81
}

# Evaluate against domain thresholds
assessment = domain_evaluator.evaluate_with_thresholds(mock_scores)

print("🎯 Domain-Aware Evaluation (Compliance)\\n")
for metric, details in assessment.items():
    if metric not in ["overall_passed", "failures"]:
        status = "✅" if details["passed"] else "❌"
        print(f"{status} {metric:20s}: {details['score']:.2f} (threshold: {details['threshold']:.2f})")

print(f"\\nOverall: {'✅ PASSED' if assessment['overall_passed'] else '❌ FAILED'}")

if assessment['failures']:
    print("\\n⚠️  Failures:")
    for failure in assessment['failures']:
        print(f"  - {failure['metric']}: {failure['severity']} severity")

# Expected: Shows pass/fail for each metric with domain-specific thresholds

### Step 3: Domain-Aware Evaluation

Different domains have different criticality requirements. A compliance system needs higher faithfulness (no hallucinations) while a customer support system needs higher relevancy (must be on-point).

**Domain-specific thresholds:**
- **Compliance:** faithfulness ≥ 0.90 (no hallucinations acceptable)
- **Customer Support:** answer_relevancy ≥ 0.85 (must be on-point)
- **General:** all metrics ≥ 0.70 (balanced)

In [ ]:
# Step 2: Run RAGAS evaluation
# Load golden set
loaded_questions = manager.load_golden_set("demo_set", "v1")

# Simulate RAG system responses (in production, these come from your RAG system)
simulated_questions = [q["question"] for q in loaded_questions]
simulated_answers = [q["ground_truth"] for q in loaded_questions]  # Perfect answers for demo
simulated_contexts = [q["contexts"] for q in loaded_questions]
ground_truths = [q["ground_truth"] for q in loaded_questions]

# Create evaluator
evaluator = RAGASEvaluator(model_name="gpt-3.5-turbo")

# Run evaluation (will skip if no API key)
results = evaluator.evaluate_system(
    questions=simulated_questions,
    generated_answers=simulated_answers,
    retrieved_contexts=simulated_contexts,
    ground_truths=ground_truths,
    skip_if_no_key=True  # Gracefully skip if no OpenAI key
)

# Print report
print(evaluator.generate_report(results))

# Expected: Either evaluation results with 4 scores, or skip message if no API key

**SAVED_SECTION:4**

### Step 2: RAGAS Evaluation

Now we integrate RAGAS metrics to evaluate our RAG system responses. In a real scenario, you would:
1. Load your golden test set
2. Query your RAG system for each question
3. Run RAGAS evaluation on the results

For this demo, we'll simulate RAG responses to show how evaluation works.

In [ ]:
# Step 1: Create a golden test set
manager = GoldenSetManager()

# Create sample questions
questions = []

# Question 1: Compliance domain
questions.append(manager.create_question(
    question="What are the GDPR data retention requirements for employee records in healthcare?",
    ground_truth="Under GDPR Article 17 and healthcare-specific regulations, employee records must be retained for 6 years after employment ends, or longer if required by national healthcare record laws. Medical information within employee records may require retention up to 8 years.",
    contexts=[
        "GDPR Article 17 establishes the right to erasure but includes exemptions for legal obligations.",
        "Healthcare employment records combine general employment data (6 year retention) with medical clearances.",
        "National laws may impose longer retention periods than GDPR minimums."
    ],
    metadata={"category": "data_retention", "complexity": "high"}
))

# Question 2: Incident response
questions.append(manager.create_question(
    question="If a contractor violates SOC 2 requirements, what are our notification obligations?",
    ground_truth="Within 72 hours of discovering contractor violation of SOC 2 controls, notification required to: affected customers, audit committee, and external auditor. Documentation must include scope of violation, affected controls, and remediation timeline.",
    contexts=[
        "SOC 2 compliance requires continuous monitoring of vendor and contractor security controls.",
        "The 72-hour notification requirement applies to violations that impact security, availability, processing integrity, confidentiality, or privacy controls.",
        "Notification must include: description of violation, affected systems/data, list of impacted controls, root cause analysis, and remediation plan."
    ],
    metadata={"category": "incident_response", "complexity": "medium"}
))

# Save the golden set
filepath = manager.save_golden_set(questions, "demo_set", "v1")
print(f"\n✅ Golden set created: {filepath}")

# Expected: Golden set saved with 2 questions, validation statistics shown

# Module 8.1: RAGAS Evaluation Framework

**Duration:** 40 minutes  
**Level:** 2 (Advanced RAG Systems)  
**Prerequisites:** Level 1 M4.3 (Basic metrics validation), working RAG system with monitoring

---

## Objectives

By the end of this notebook, you will:

- Implement RAGAS framework for systematic RAG evaluation with 4 metrics: faithfulness, answer relevance, context precision, and context recall
- Create and maintain a golden test set of 100+ questions with ground truth answers
- Build automated nightly evaluation pipelines that detect regressions before production
- Track RAG performance over time with baseline comparisons and statistical significance testing
- **Decide when NOT to use RAGAS** and what simpler alternatives exist

---

## Section 1: Introduction & Hook

### The Problem with "Trust but Don't Verify"

In Level 1 M4.3, you built basic validation checks for your RAG system. You're comparing outputs to expected answers, tracking cache hit rates, maybe logging examples. It works... until it doesn't.

**Real production scenario:**
- You deploy a prompt optimization that 'feels better' in testing
- Within 48 hours, users report incorrect compliance citations
- The system is hallucinating regulation numbers that don't exist
- Your basic checks? They passed. Why? Because you were only checking semantic similarity, not factual accuracy.

**The production gap:**
You're making dozens of changes—new embeddings, different chunking strategies, model updates, prompt tweaks. Each affects answer quality in subtle ways. But you have no systematic way to know if you're improving or degrading.

**How do you:**
- Evaluate RAG system quality systematically?
- Detect regressions before users complain?
- Build confidence that your improvements actually improve things?

In [ ]:
# Setup: Import required modules
import sys
sys.path.append('.')

from l2_m8_ragas_evaluation_framework import (
    GoldenSetManager,
    RAGASEvaluator,
    DomainAwareEvaluator,
    ResilientEvaluator,
    EvaluationPipeline
)
from config import Config

print("✅ Imports successful")
print(f"OpenAI configured: {Config.has_openai_key()}")

# Expected: Module imports work, config shows if OpenAI key is set

**SAVED_SECTION:1**

---

## Section 2: Prerequisites & Setup

### Starting Point Verification

Your Level 1 system currently has some validation, probably something like:

```python
# Current approach from Level 1 M4.3
def validate_response(query: str, response: str, expected: str) -> bool:
    # Simple semantic similarity check
    response_embedding = get_embedding(response)
    expected_embedding = get_embedding(expected)
    similarity = cosine_similarity(response_embedding, expected_embedding)
    return similarity > 0.75  # Threshold-based pass/fail
```

**The gaps in this approach:**
1. **No factual grounding check:** High similarity ≠ factually correct
2. **No retrieval quality assessment:** You're not checking if you retrieved the right documents
3. **No systematic tracking:** Pass/fail rates aren't trended over time
4. **No regression detection:** You can't tell if today is worse than yesterday

**Real example:**
87% semantic similarity on a compliance query. Response was beautifully written, contextually similar... and cited the wrong regulation. RAGAS faithfulness metric? 0.12/1.0. Would have caught it immediately.

By the end of today, you'll have:
- Four metrics (not one)
- 100+ test cases (not 5)
- Automated nightly runs (not manual)
- Baseline tracking (not isolated checks)

In [ ]:
# Validate environment setup
Config.validate()

print("\n📋 Configuration:")
print(f"  Golden Set Dir: {Config.GOLDEN_SET_DIR}")
print(f"  Results Dir: {Config.RESULTS_DIR}")
print(f"  Domain: {Config.DOMAIN}")
print(f"  Batch Size: {Config.BATCH_SIZE}")

print("\n📊 Thresholds:")
for metric, threshold in Config.get_domain_thresholds().items():
    print(f"  {metric}: {threshold:.2f}")

# Expected: Configuration validated, directories exist, thresholds shown

### Dependencies Check

RAGAS requires:
- `ragas==0.1.8` - Core RAGAS framework
- `langchain==0.1.20` - LangChain integration
- `datasets==2.16.1` - HuggingFace datasets for RAGAS format
- `openai>=1.12.0` - LLM-as-judge (GPT-3.5-Turbo or GPT-4)
- `mlflow==2.9.2` - (Optional) Performance tracking

**Cost warning:**
RAGAS uses GPT-3.5-Turbo or GPT-4 as a judge. Evaluating 100 questions costs ~$2-5 depending on model. This is ongoing operational cost.

In [ ]:
# Check if RAGAS is installed
try:
    import ragas
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    print(f"✅ RAGAS version: {ragas.__version__}")
except ImportError:
    print("⚠️  RAGAS not installed. Run: pip install -r requirements.txt")

# Check OpenAI access
if Config.has_openai_key():
    print("✅ OpenAI API key configured")
else:
    print("⚠️  OpenAI API key not configured - evaluation will skip API calls")
    print("   Set OPENAI_API_KEY in .env file")

# Expected: Shows RAGAS version and OpenAI configuration status

**SAVED_SECTION:2**

---

## Section 3: Theory Foundation

### Understanding RAGAS: RAG Assessment Framework

Before we code, let's understand what RAGAS actually measures and why it matters.

**The core insight:**
Traditional NLP metrics like BLEU or ROUGE don't work for RAG systems. Why? Because RAG systems can generate correct answers in many different ways—different phrasing, different structure, different supporting evidence—all equally valid.

**RAGAS provides four complementary metrics:**

#### 1. Faithfulness (0-1 score)
- **What it measures:** Is the answer grounded in the retrieved context, or is it hallucinating?
- **How it works:** Breaks down the answer into statements, then checks if each statement can be inferred from retrieved chunks
- **Example:** If answer says 'Revenue grew 42%' but context says 'Revenue grew 35%', faithfulness drops
- **Why it matters:** Catches hallucinations that semantic similarity misses

#### 2. Answer Relevancy (0-1 score)
- **What it measures:** Does the answer actually address what was asked?
- **How it works:** Generates variations of questions from the answer, then checks similarity to original question
- **Why it matters:** Catches tangential responses that are factually correct but miss the point

#### 3. Context Precision (0-1 score)
- **What it measures:** Are the most relevant chunks ranked highest in your retrieval results?
- **How it works:** Checks if relevant chunks appear at top positions
- **Why it matters:** Identifies retrieval ranking issues (hybrid search misconfiguration, bad reranking)

#### 4. Context Recall (0-1 score)
- **What it measures:** Did you retrieve all the chunks needed to answer the question?
- **How it works:** Compares ground truth answer against retrieved chunks
- **Why it matters:** Catches retrieval gaps (missing key documents in index, bad query transformation)

### RAGAS Evaluation Flow

```
User Query
    ↓
Your RAG System
    ↓
Retrieved Chunks (contexts)
    ↓
Generated Answer
    ↓
RAGAS Metrics
├── Faithfulness: Answer vs Contexts
├── Answer Relevancy: Answer vs Query
├── Context Precision: Contexts ranking vs Ground truth
└── Context Recall: Contexts vs Ground truth
    ↓
Scores (0-1 for each)
```

**How RAGAS works under the hood:**
RAGAS uses GPT-3.5-Turbo (or GPT-4) as an 'LLM judge' to evaluate your responses. Research shows LLM judges correlate 85-90% with human judgments when properly prompted.

**Common misconception:**
'RAGAS gives me one score to optimize.' FALSE. You get four scores because RAG has four distinct failure modes:

- **Faithfulness drop** → You're hallucinating more (prompt change broke something)
- **Answer relevancy drop** → You're being verbose but not helpful (system message issue)
- **Context precision drop** → Your reranking is broken or hybrid search alpha is wrong
- **Context recall drop** → You're missing key documents (indexing gaps, query transformation issues)

Different problems, different fixes. That's the power of RAGAS.

In [ ]:
# Demonstrate the four metrics conceptually
print("📊 RAGAS Four Metrics Overview\n")

metrics_info = [
    ("Faithfulness", "Is answer grounded in context?", "Catches hallucinations"),
    ("Answer Relevancy", "Does answer address the question?", "Catches tangential responses"),
    ("Context Precision", "Are relevant chunks ranked high?", "Identifies ranking issues"),
    ("Context Recall", "Did we retrieve everything needed?", "Catches retrieval gaps")
]

for name, question, purpose in metrics_info:
    print(f"✓ {name:20s}")
    print(f"  {question}")
    print(f"  → {purpose}\n")

# Expected: Prints overview of 4 RAGAS metrics

**SAVED_SECTION:3**

---

## Section 4: Hands-on Implementation

### Step 1: Creating Your Golden Test Set

The foundation of any evaluation system is your golden test set—a curated collection of questions with ground truth answers that represent real user queries.

**What makes a good golden test set:**
- 100-300 questions (100 minimum for statistical significance)
- Covers all major query types in your domain
- Includes edge cases and known failure modes
- Has ground truth answers reviewed by domain experts
- Updated quarterly as your system evolves

**Key insight:**
Golden set creation is HARD WORK. This is 40-80 hours of domain expert time to create 100-300 questions. You can't skip this. Bad golden set = worthless evaluation.